In [ ]:
# =========================================================
# INSTALL LIBRARIES
# =========================================================
!pip -q install xgboost torch scikit-learn pandas numpy gradio


# =========================================================
# IMPORTS
# =========================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import gradio as gr


# =========================================================
# LSTM MODEL (PyTorch)
# =========================================================
class LSTMLayer(nn.Module):
    def __init__(self, input_size=8, hidden_size=32):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        out = self.fc(h[-1])
        return self.sigmoid(out)



# =========================================================
# FLOOD PREDICTOR (LSTM + XGBOOST ENSEMBLE)
# =========================================================
class FloodPredictor:

    def __init__(self):

        self.scaler = StandardScaler()
        self.xgb = XGBClassifier(
            n_estimators=150,
            max_depth=5,
            learning_rate=0.1,
            random_state=42
        )

        self.lstm = LSTMLayer()


    # -------------------------
    # TRAIN MODELS
    # -------------------------
    def train(self):

        print("Training models...")

        n = 5000

        df = pd.DataFrame({
            'water_level': np.random.normal(1.5,0.6,n),
            'rainfall': np.random.exponential(20,n),
            'temp': np.random.normal(30,5,n),
            'humidity': np.random.normal(80,10,n),
            'river_discharge': np.random.exponential(70,n),
            'elevation': np.random.normal(20,8,n),
            'lat': np.full(n, 13.08),
            'lon': np.full(n, 80.27)
        })

        # Flood rule (synthetic labels)
        y = (
            (df.water_level > 2.2) &
            (df.rainfall > 30) &
            (df.river_discharge > 100)
        ).astype(int)

        X = self.scaler.fit_transform(df)

        # Train XGBoost
        self.xgb.fit(X, y)

        print("✅ Training complete")


    # -------------------------
    # PREDICT
    # -------------------------
    def predict(self, features):

        df = pd.DataFrame([features])
        X = self.scaler.transform(df)

        # XGBoost
        xgb_prob = self.xgb.predict_proba(X)[0][1]

        # LSTM
        tensor = torch.FloatTensor(X).unsqueeze(0)

        with torch.no_grad():
            lstm_prob = self.lstm(tensor).item()

        # Ensemble
        final = 0.9*xgb_prob + 0.1*lstm_prob


        return final, xgb_prob, lstm_prob



# =========================================================
# TRAIN MODEL
# =========================================================
model = FloodPredictor()
model.train()



# =========================================================
# GRADIO WEB APP (Creates public link automatically)
# =========================================================
def predict_app(water, rain, temp, humidity, discharge, elevation):

    data = {
        "water_level": water,
        "rainfall": rain,
        "temp": temp,
        "humidity": humidity,
        "river_discharge": discharge,
        "elevation": elevation,
        "lat": 13.08,
        "lon": 80.27
    }

    final, xgb, lstm = model.predict(data)

    msg = f"""
Flood Probability: {final*100:.2f}%

XGBoost: {xgb*100:.2f}%
LSTM: {lstm*100:.2f}%
"""

    if final > 0.7:
        msg += "\n🚨 FLOOD ALERT"
    else:
        msg += "\n✅ SAFE"

    return msg
# =========================================================
# UI
# =========================================================
demo = gr.Interface(
    fn=predict_app,
    inputs=[
        gr.Slider(0,3,label="Water Level (m)"),
        gr.Slider(0,60,label="Rainfall (mm)"),
        gr.Slider(10,45,label="Temperature (°C)"),
        gr.Slider(40,100,label="Humidity (%)"),
        gr.Slider(0,200,label="River Discharge"),
        gr.Slider(0,50,label="Elevation")
    ],
    outputs="text",
    title="🌊 LSTM + XGBoost Flood Prediction System",
    description="Advanced ML ensemble model running fully in Google Colab"
)


demo.launch(share=True)






Training models...
✅ Training complete
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ab588b3611f66cf3cd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
